# CIFAR-10 Image Transformation Pipeline Demo

This notebook demonstrates the image transformation pipeline concepts introduced in class.

Steps:
1. Define training and test pipelines (preprocessing → augmentation → normalization)
2. Load CIFAR-10 dataset with these transforms
3. Visualize augmented samples
4. (Optional) Define a simple CNN model
5. (Optional) Train for a few epochs (not executed here)

## 1. Imports

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

# Ensure reproducibility
torch.manual_seed(42)

## 2. Define Transformation Pipelines

In [ ]:
# Training pipeline (preprocessing → augmentation → normalization)
train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),                  # preprocessing
    transforms.RandomCrop(32, padding=4),         # geometric augmentation
    transforms.RandomHorizontalFlip(),            # geometric augmentation
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # photometric
    transforms.ToTensor(),                        # convert to tensor
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)) # normalize
])

# Test pipeline (deterministic only)
test_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

## 3. Load CIFAR-10

In [ ]:
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=train_transforms)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=test_transforms)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

## 4. Visualize Augmented Samples

In [ ]:
def imshow(img):
    img = img / 2 + 0.5  # unnormalize (approx)
    npimg = img.numpy().transpose((1,2,0))
    plt.figure(figsize=(6,6))
    plt.imshow(npimg)
    plt.axis('off')

# get a batch of training data
dataiter = iter(train_loader)
images, labels = next(dataiter)

# show images
imshow(make_grid(images[:16], nrow=4))
plt.show()

## 5. (Optional) Define a Simple CNN Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## 6. (Optional) Training Loop (Not Executed Here)

In [ ]:
# Example training loop structure (not executed)
model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

print('Training complete (demo only)')